To start: srun --partition=hpg-b200 --mem=16gb --gpus=1 --time=05:00:00 --pty bash -i
module load jupyter
launch_jupyter_notebook

In [13]:
!ls *.tex

ndr-30-agga-avg.tex


In [5]:
!ls s_*

s_bl_0.pt  s_bl_2.pt  s_bl_4.pt  s_bl_6.pt  s_bl_8.pt
s_bl_1.pt  s_bl_3.pt  s_bl_5.pt  s_bl_7.pt  s_bl_9.pt


In [6]:
import torch

In [7]:
s = torch.load('s_bl_0.pt')

In [10]:
!cat ../groups.json

{ "v-b-8": ".*\\.layers\\.8\\..*\\.v_proj\\..*\\.lora_B\\..*",
  "v-b-9": ".*\\.layers\\.9\\..*\\.v_proj\\..*\\.lora_B\\..*"
}


In [21]:
test = 'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight'

In [30]:
import re
p = re.compile(".*\\.layers\\.8\\..*\\.v_proj\\.lora_B\\..*")

In [33]:
x = p.match(test)
x

<re.Match object; span=(0, 70), match='base_model.model.model.layers.8.self_attn.v_proj.>

In [19]:
[k for k in s.keys() if "layers.8." in k[2] and 'lora_B' in k[2] and 'v_proj' in k[2]]

[('datainf',
  'mean',
  'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight'),
 ('datainf',
  'rank-c',
  'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight'),
 ('datainf',
  'vote2-c',
  'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight'),
 ('cos',
  'mean',
  'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight'),
 ('cos',
  'rank-c',
  'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight'),
 ('cos',
  'vote2-c',
  'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight'),
 ('hf',
  'mean',
  'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight'),
 ('hf',
  'rank-c',
  'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight'),
 ('hf',
  'vote2-c',
  'base_model.model.model.layers.8.self_attn.v_proj.lora_B.default.weight')]

In [3]:
%cd /blue/anshumanc.usf/nn-infl/llama/qnli

/blue/anshumanc.usf/nn-infl/llama/qnli


In [25]:
from collections import defaultdict
from functools import partial
import json
import os
from matplotlib.colors import ListedColormap
from matplotlib.ticker import FuncFormatter, MultipleLocator
import pandas as pd
import re
from typing import Literal, Optional
from tabulate import tabulate

import datasets
import matplotlib
from matplotlib import pyplot as plt
import numpy as np
from pandas import DataFrame
from scipy import stats
import torch
from datasets import load_from_disk
from scipy import stats as sci_stats
from sklearn.preprocessing import StandardScaler

# from pyclustering.cluster.xmeans import xmeans
from sklearn.metrics import silhouette_samples
# import warnings
# np.warnings = warnings

matplotlib.rcParams['text.usetex'] = False
#matplotlib.rcParams['text.latex.preamble'] = r"\usepackage{times}"
#matplotlib.rcParams['pdf.fonttype'] = 42
#matplotlib.rcParams['ps.fonttype'] = 42
#matplotlib.rcParams['font.family'] = ['Times']   
benchmark = ["qnli", "mrpc", "sst2", "qqp", "cola", "mnli", "rte", "stsb"]

In [3]:
def get_df_from_file(metric_file: str):    

    # file_name = os.path.basename(metric_file).split(".")[0]
    with open(metric_file, 'r') as f:
        runs = [json.loads(l) for l in f.readlines()]

    rows = []
    rand_accuracies = {}
    for r in runs: 

        accuracy0 = r["first_finetune"]["accuracy"]
        accuracy = r["accuracy"]
        best_accuracy0 = np.max(accuracy0)
        best_accuracy = np.max(accuracy)
        accuracy_delta = best_accuracy - best_accuracy0

        infl_accuracy0 = r["first_finetune"]["infl_accuracy"]
        infl_accuracy = r["infl_accuracy"]
        best_infl_accuracy0 = np.max(infl_accuracy0)
        best_infl_accuracy = np.max(infl_accuracy)
        infl_accuracy_delta = best_infl_accuracy - best_infl_accuracy0

        infl_loss0 = r["first_finetune"]["infl_loss"]
        infl_loss = r["infl_loss"]
        min_infl_loss0 = np.min(infl_loss0)
        min_infl_loss = np.min(infl_loss)        
        infl_loss_delta = min_infl_loss - min_infl_loss0

        # logits_change = r["logits_change"]

        auc_ndr = r["auc_ndr"]
        filtered = r["filtered"]

        noise_curve = r["ndr_curve"]
        num_noise = round(0.2 * len(noise_curve))
        ideal_area = num_noise / 2 + (len(noise_curve) - num_noise)        
        auc_ndr2 = sum((noise_curve[i] + noise_curve[i + 1]) / 2 for i in range(len(noise_curve) - 1)) / ideal_area

        task = r["config"]["task"]
        infl_method = r["config"]["infl_method"]
        seed = r["config"]["seed"]
        filter_perc = r["config"]["filter_perc"]

        index_30 = round(filter_perc * len(noise_curve))
        noise_30  = noise_curve[index_30]

        assert np.allclose(auc_ndr, auc_ndr2, atol=1e-2)
        assert np.allclose(noise_30, filtered, atol=1e-2)

        if infl_method == "rand":
            rand_accuracies[(task, filter_perc, seed)] = (best_accuracy, best_infl_accuracy)

        module_name = r["config"]["module_name"]

        row = {
               "task": task,
               "filter_perc": filter_perc,
               "infl_method": infl_method,
               "agg_method": r["config"]["agg_method"],
               "module": module_name,
               
               "best_accuracy_delta": accuracy_delta,
               "best_infl_accuracy_delta": infl_accuracy_delta,
                "infl_loss_delta": infl_loss_delta,
            #    "logits_change": logits_change,
               "noise_30": filtered,
               "auc_ndr": auc_ndr,

               "best_accuracy_0": best_accuracy0,
               "best_accuracy_1": best_accuracy,

               "best_infl_accuracy_0": best_infl_accuracy0,
               "best_infl_accuracy_1": best_infl_accuracy,

                "infl_loss_0": min_infl_loss0,
                'infl_loss_1': min_infl_loss,

               "seed": seed

               }
        rows.append(row)

    if len(rand_accuracies) > 0:
        for r in rows:
            key = (r["task"], r["filter_perc"], r["seed"])
            rand_accuracy, rand_infl_accuracy = rand_accuracies[key]
            accuracy_rand_delta = r["best_accuracy_1"] - rand_accuracy
            infl_accuracy_rand_delta = r["best_infl_accuracy_1"] - rand_infl_accuracy
            r["accuracy_rand_delta"] = accuracy_rand_delta
            r["infl_accuracy_rand_delta"] = infl_accuracy_rand_delta

    df = pd.DataFrame(rows)

    return df

def get_all_df(base_path = "data/roberta/filter-30", datasets = ["mrpc", "qnli", "sst2", "qqp", "cola", "mnli", "rte", "stsb"],
                res_suffix = "bl", keep_only = None):
    # if os.path.exists(tmp_file):
    #     df1 = pd.read_pickle(tmp_file)
    # else:
    keep_only = ["task", "infl_method", "agg_method", "module", "seed", *keep_only]
    plain_dataframes = []
    for dataset in datasets:
        file = os.path.join(base_path, "metrics",  f"{dataset}-{res_suffix}.jsonlist")
        df = get_df_from_file(file)
        if keep_only is not None:
            df.drop(columns=[c for c in df.columns if c not in keep_only], inplace=True)
        plain_dataframes.append(df)
    
    df1 = pd.concat(plain_dataframes, ignore_index=True)
    # df1.to_pickle(tmp_file)
    return df1

def load_df(base_path, tasks = benchmark, use_ndr = False,
                ndr_prefix = "ndr_bl", agg_method_names = None,
                infl_method_names = None, selected_layers = None,
                res_suffix = "all", metric_name = "best_accuracy_1"):
    # loading data 
    if use_ndr: 
        dfs = []
        for task in tasks:
            df = pd.read_pickle(os.path.join(base_path, f"{ndr_prefix}_{task}.pcl"))
            df.drop(columns=["scores", "noise_mask"], inplace=True)
            df.reset_index(inplace=True)
            if agg_method_names is not None:
                df = df[df["agg"].isin(agg_method_names)]
            if infl_method_names is not None:
                df = df[df["infl"].isin(infl_method_names)]
            if selected_layers is not None:
                df = df[df["layer"].isin(selected_layers)]
            dfs.append(df)
        df = pd.concat(dfs, ignore_index=True)
        df = df.rename(columns={"run_id": "seed"})
        df = df[["task", "infl", "agg", "layer", "module", "seed", metric_name]]

        def layer_module_normalize(row):
            layer = row["layer"]
            module = row["module"].replace("embed_tokens", "WE")
            find_layer = re.match(r"layers\s(\d+)\s", module)
            if find_layer is not None:
                layer = find_layer.group(1)
                module = module.replace(find_layer.group(0), "")
            row["layer"] = "WE" if module == "WE" else layer
            row["module"] = module
            return row

        df = df.apply(layer_module_normalize, axis=1)
        
    else: #use jsonlists 
        df = get_all_df(base_path = base_path, datasets = tasks,
                                res_suffix = res_suffix, keep_only=[metric_name])
        if agg_method_names is not None:
            df = df[df["agg_method"].isin(agg_method_names)]
        if infl_method_names is not None:
            df = df[df["infl_method"].isin(infl_method_names)]
        if selected_layers is not None:
            df = df[df["module"].isin(selected_layers)]        
        df = df.rename(columns={"infl_method": "infl", "agg_method": "agg", "module": "layer"})
        df["module"] = "all"
    return df

In [4]:
%cd /blue/anshumanc.usf/nn-infl/llama

/blue/anshumanc.usf/nn-infl/llama


In [5]:
def get_confidence_interval(data_df: pd.DataFrame, confidence_level=0.95):
    data_np_t = data_df.to_numpy().T
    data_mean = data_np_t.mean(axis=0)
    degrees_freedom = data_np_t.shape[0] - 1 # num of measures - 1
    sample_standard_error = stats.sem(data_np_t, axis=0)
    confidence_interval = stats.t.interval(confidence_level, degrees_freedom, data_mean, sample_standard_error)
    min_v = confidence_interval[0]
    max_v = confidence_interval[1]
    res_df = pd.DataFrame(data={"min_v": min_v, "mean": data_mean, "max_v": max_v}, index=data_df.index)
    return res_df

In [19]:
def layer_ndr_metric_graphs(base_path: str, tasks=benchmark, 
                    metric_name = 30, figsize = (8, 2),
                    infl_method_names = ['datainf', 'hf', 'cos'],
                    module_names = {"self_attn q_proj A": "Query A", "self_attn q_proj B": "Query B", 
                                    "self_attn v_proj A": "Value A", "self_attn v_proj B": "Value B"},
                    # module_names = {"self_attn q_proj A": "query A"},
                    module_layers = list(range(32)),
                    cl_modules = {"all": "CL"}, # makes sense for Roberta only
                    ndr_prefix = "ndr_bl", suffix = "",
                    max_of = "self_attn v_proj B",
                    y_title = "Layer-wise NDR, \\%"):
    ''' From all run scores, computes the interaction matrix between setups  
        and their pareto fronts 
    '''
    orig_infl_names = list(infl_method_names)

    cache_path = os.path.join(base_path, f"cache_layer_ndr{suffix}.pcl")
    if os.path.exists(cache_path):
        score_df = pd.read_pickle(cache_path) 
    else:
        df = load_df(base_path, tasks, use_ndr = True, ndr_prefix = ndr_prefix, metric_name=metric_name, 
                    agg_method_names=['mean'], infl_method_names=infl_method_names)
    
        score_df = df.pivot(index=["infl", "agg", "layer", "module"], columns=["task", "seed"], values=metric_name)
        score_df.to_pickle(cache_path)

    score_df *= 100.0

    infl_method_names = orig_infl_names


    # mean_df = score_df.mean(axis=1)
    conf_int_df = get_confidence_interval(score_df)
    # std_df = score_df.std(axis=1)

    very_min = conf_int_df.min(axis=1).min()
    very_max = conf_int_df.max(axis=1).max()

    module_name_list = list(module_names.keys())


    ndr_per_infl_values = []
    infl_method_mapping = {"datainf": "DataInf", "hf": "TracIn", "cos": "Cosine"}
    for infl_method in infl_method_names:
        infl_name = infl_method_mapping[infl_method]
        infl_chart = {"id": infl_method, "name": infl_name}
        x_markers = infl_chart.setdefault("x_markers", [])
        end_x_marks = []
        # x_min_values = []
        # x_mean_values = []
        # x_max_values = []
        # x_markers = [] 
        # common_start = []
        # common_end = []
        we_value = None
        if infl_method == "hf": 
            start_y_min_values = infl_chart.setdefault("start_y_min_values", [])
            start_y_mean_values = infl_chart.setdefault("start_y_mean_values", [])
            start_y_max_values = infl_chart.setdefault("start_y_max_values", [])   
            we_value = conf_int_df.loc[("hf", "mean", "WE", "all")]
            start_y_min_values.append(we_value["min_v"])
            start_y_mean_values.append(we_value["mean"])
            start_y_max_values.append(we_value["max_v"])
            x_markers.append("WE")
        elif infl_method == "cos":
            start_y_min_values = infl_chart.setdefault("start_y_min_values", [])
            start_y_mean_values = infl_chart.setdefault("start_y_mean_values", [])
            start_y_max_values = infl_chart.setdefault("start_y_max_values", [])
            we_value = conf_int_df.loc[("cos", "mean", "WE", "all")]
            start_y_min_values.append(we_value["min_v"])
            start_y_mean_values.append(we_value["mean"])
            start_y_max_values.append(we_value["max_v"])
            x_markers.append("WE")
        prev_we_value = we_value
        next_we_value = None
        for cl_module in cl_modules.keys():
            end_y_min_values = infl_chart.setdefault("end_y_min_values", [])
            end_y_mean_values = infl_chart.setdefault("end_y_mean_values", [])
            end_y_max_values = infl_chart.setdefault("end_y_max_values", [])
            we_value = conf_int_df.loc[(infl_method, "mean", "CL", cl_module)]
            end_y_min_values.append(we_value["min_v"])
            end_y_mean_values.append(we_value["mean"])
            end_y_max_values.append(we_value["max_v"])
            end_x_marks.append(cl_modules[cl_module])
            if next_we_value is None:
                next_we_value = we_value
        if prev_we_value is not None:
            for module_name in module_name_list:
                y_min_values = infl_chart.setdefault(f"{module_name}:min", [])
                y_mean_values = infl_chart.setdefault(f"{module_name}:mean", [])
                y_max_values = infl_chart.setdefault(f"{module_name}:max", [])
                y_min_values.append(prev_we_value["min_v"])
                y_mean_values.append(prev_we_value["mean"])
                y_max_values.append(prev_we_value["max_v"])                
        for module_layer in module_layers:
            layer_name = str(module_layer)
            x_markers.append(module_layer + 1)
            for module_name in module_name_list:
                y_min_values = infl_chart.setdefault(f"{module_name}:min", [])
                y_mean_values = infl_chart.setdefault(f"{module_name}:mean", [])
                y_max_values = infl_chart.setdefault(f"{module_name}:max", [])
                we_value = conf_int_df.loc[(infl_method, "mean", layer_name, module_name)]
                y_min_values.append(we_value["min_v"])
                y_mean_values.append(we_value["mean"])
                y_max_values.append(we_value["max_v"])
        if next_we_value is not None:
            for module_name in module_name_list:
                y_min_values = infl_chart.setdefault(f"{module_name}:min", [])
                y_mean_values = infl_chart.setdefault(f"{module_name}:mean", [])
                y_max_values = infl_chart.setdefault(f"{module_name}:max", [])
                y_min_values.append(next_we_value["min_v"])
                y_mean_values.append(next_we_value["mean"])
                y_max_values.append(next_we_value["max_v"]) 
        x_markers.extend(end_x_marks)
        ndr_per_infl_values.append(infl_chart)

    plt.ioff()
    fig, axes = plt.subplots(1, len(ndr_per_infl_values), figsize=figsize)
    ordered_handles = []
    ordered_labels = []
    for chart_id, chart_data in enumerate(ndr_per_infl_values):
        ax =  axes[chart_id]
        start_id = 1
        if "start_y_mean_values" in chart_data:
            start_means = chart_data["start_y_mean_values"]
            start_id += (len(start_means) - 1)
        if start_id > 1:
            adjustment = [1, 3.5, 6]
            start_id = adjustment[-1]
        else:
            adjustment = [1]
        x_max_pos = None
        for module_name in module_name_list:
            y_min_values = chart_data[f"{module_name}:min"]
            y_mean_values = chart_data[f"{module_name}:mean"]
            y_max_values = chart_data[f"{module_name}:max"]
            local_x = np.arange(len(y_mean_values)) + start_id
            if max_of == module_name:
                max_pos = np.argmax(y_mean_values)
                x_max_pos = local_x[max_pos]
            ln = ax.plot(local_x, y_mean_values, label=module_names[module_name], linewidth=1)
            if chart_id == 0:
                ordered_handles.append(ln[0])
                ordered_labels.append(module_names[module_name])
            ax.fill_between(local_x, y_min_values, y_max_values, color=ln[0].get_color(), alpha=0.2)
            # start_id += len(y_mean_values)
        start_id += (len(y_mean_values) - 1)
        if "end_y_mean_values" in chart_data:
            end_means = chart_data["end_y_mean_values"]
            local_x = np.arange(len(end_means)) + start_id
            ln = ax.plot(local_x, end_means, color="black", linewidth=1) 
            ax.fill_between(local_x, chart_data["end_y_min_values"], chart_data["end_y_max_values"], color=ln[0].get_color(), alpha=0.2)           
            # start_id += len(end_means)
        start_id = 1
        if "start_y_mean_values" in chart_data:
            start_means = chart_data["start_y_mean_values"]
            local_x = np.arange(len(start_means)) + start_id
            ln = ax.plot(adjustment, start_means, color="black", linewidth=1) #linestyle="--", linewidth=1.5)
            if len(adjustment) > 1 and chart_id == 1:
                ordered_handles.append(ln[0])
                ordered_labels.append("WE methods")
            ax.fill_between(adjustment, chart_data["start_y_min_values"], chart_data["start_y_max_values"], color=ln[0].get_color(), alpha=0.2)
            # start_id += len(start_means)            
        x_markers = chart_data["x_markers"]
        tick_poss = adjustment + list(np.arange(len(x_markers) - len(adjustment)) + 1 + adjustment[-1])
        ax.set_xlim(0.5, (adjustment[-1] + len(x_markers) - len(adjustment))+0.5)
        filtered_labels = []
        filtered_ticks = []
        for mid, m in enumerate(x_markers):
            try:
                num = int(m)
                if num % 3 == 0:
                    filtered_ticks.append(tick_poss[mid])
                    filtered_labels.append(str(num))
                else:
                    if len(filtered_labels) == 0:
                        filtered_ticks.append(tick_poss[mid])
                        filtered_labels.append(str(num))
                    # else:
                    #     filtered_labels.append("")
            except ValueError:
                filtered_ticks.append(tick_poss[mid])
                filtered_labels.append(m)
        ax.set_xticks(filtered_ticks)
        ax.set_xticklabels(filtered_labels, fontsize=7, verticalalignment='center')
        ax.tick_params(axis='x', length=1)
        ax.tick_params(axis='y', pad=0, length=1)
        if chart_id == 0:
            ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{int(value)}"))
            ax.set_ylabel(y_title, fontsize=10, labelpad=0)
            ax.set_yticks([40,50,60,70,80,90])
        else:
            ax.set_yticks([])
        ax.set_yticklabels(ax.get_yticks(), fontsize=7) 
        ax.set_ylim(very_min, very_max)
        ax.set_xlabel("Layers", fontsize=8, labelpad=0)
        ax.set_title(chart_data["name"], fontsize=10, pad=0)
        for yhline in [40,50,60,70,80,90]:
            ax.axhline(y=yhline, color="gray", linewidth=0.5, linestyle="--")
        if x_max_pos is not None:
            ax.axvline(x=x_max_pos, color="gray", linewidth=0.5, linestyle="--")

    fig.legend(ordered_handles, ordered_labels, loc='lower center', fontsize=8,
                ncol=len(ordered_handles), borderaxespad = 0,  # Arrange all legend items in one row
                bbox_to_anchor=(0.5, 0)  # Adjust position (centered below the grid)
        )        
    fig.tight_layout(w_pad=0, h_pad=0, pad=0, rect=[0, 0.13, 1, 1])
    fig.savefig(os.path.join(base_path, f"layers-ndr-{suffix}.png"))
    plt.close(fig)
    pass


In [20]:
network_layers = {
    "roberta": ['WE', '00-05', '06-11', '12-17', '18-23', 'CL'],
    "llama": ['WE', '00-03', '04-07', '08-11', '12-15', 'CL'],
    "mistral": ['WE', '00-07', '08-15', '16-23', '24-31', 'CL'],
    "qwen": ['WE', '00-06', '07-13', '14-20', '21-27', 'CL'],
}

network_modules = {
    "roberta": {"query A": "Query A", "query B": "Query B", 
                "value A": "Value A", "value B": "Value B"},
    "llama": {"self_attn q_proj A": "Query A", "self_attn q_proj B": "Query B", 
                "self_attn v_proj A": "Value A", "self_attn v_proj B": "Value B"},
    "qwen": {"self_attn q_proj A": "Query A", "self_attn q_proj B": "Query B", 
                "self_attn v_proj A": "Value A", "self_attn v_proj B": "Value B"},
    "mistral": {"self_attn q_proj A": "Query A", "self_attn q_proj B": "Query B", 
                "self_attn v_proj A": "Value A", "self_attn v_proj B": "Value B"},                                
}


network_layer_count = {
    "roberta": 24,
    "llama": 16,
    "mistral": 32,
    "qwen": 28
}

network_best_ndr = {
    "roberta": "value B",
    "mistral": "self_attn v_proj B",
    "qwen": "self_attn v_proj B",
    "llama": "self_attn v_proj B",
}

In [26]:
# network = "mistral"
# network="roberta"
network="llama"
# network="qwen"
group_file = "./groups.json"
base_path = f"./"
selected_layers = network_layers[network]
noise_only = False

model_num_layers = {
    "roberta": 24,
    "qwen": 28
}

infl_method_names = ["datainf", "hf", "cos"] #, "denoise", "rand"]
agg_method_names = ["mean", '']
res_suffix = "bl"
# setup_interactions(base_path, use_ndr=False, metric_name="best_accuracy_1",
#                     agg_method_names = agg_method_names, dom_threshold = 0.5,
#                     infl_method_names=infl_method_names, suffix="-mean",
#                     res_suffix=res_suffix)    
selected_network_modules = network_modules[network]
layer_ndr_metric_graphs(base_path, metric_name=30,
                    infl_method_names=infl_method_names, suffix="",
                    module_layers = list(range(network_layer_count[network])),
                    module_names = selected_network_modules,
                    max_of = network_best_ndr[network],figsize=(8, 1.5))
pass 

findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font family 'Times' not found.
findfont: Font f

In [7]:



def process_ndr_table(base_path: str, tasks: list[str] = ["qnli"], output_ranks = False, with_row_id = False,
                        metric_name = 30, ndr_prefix = "ndr_bl", layers = None,
                        best_group_by = None, custom_suffix = "",
                        agg_method_names = None, infl_method_names = None): 
    #NOTE: metric_name in ["f30", "auc_ndr"]):

    dfs = []
    for task in tasks:
        df = pd.read_pickle(os.path.join(base_path, f"{ndr_prefix}_{task}.pcl"))
        df.drop(columns=["scores", "noise_mask"], inplace=True)
        dfs.append(df)
    df = pd.concat(dfs, ignore_index=False)

    if infl_method_names is not None:
        df = df.loc[df.index.get_level_values('infl').isin(infl_method_names)]
        pass

    if agg_method_names is not None:
        df = df.loc[df.index.get_level_values('agg').isin(agg_method_names)]
        pass

    if layers is not None:
        df = df.loc[df.index.get_level_values('layer').isin(layers)]
        df = df.loc[df.index.get_level_values('module') == "all"]
        pass


    metric_df = df.reset_index().pivot(index=["infl", "agg", "layer", "module"], columns=["task", "run_id"], values=[metric_name])

    ranks = metric_df.rank(axis = 0, ascending=False, method="average")
    
    suffix = ""
    if output_ranks:
        metric_df = ranks
        suffix = "-rank"

    metric_by_ds_mean = metric_df.groupby(level=1, axis=1).mean()
    metric_by_ds_std = metric_df.groupby(level=1, axis=1).std()
    metric_by_ds_std.columns = [c + "_std" for c in metric_by_ds_std.columns]
    metric_by_ds = pd.merge(metric_by_ds_mean, metric_by_ds_std, left_index=True, right_index=True)
    rank_columns = ["rank", "rank_std"]
    metric_by_ds["rank"] = ranks.mean(axis=1)
    metric_by_ds["rank_std"] = ranks.std(axis=1)

    if best_group_by is not None:
        best_idxs = metric_by_ds.groupby(level=best_group_by, axis=0)['rank'].idxmin()
        metric_by_ds = metric_by_ds.loc[best_idxs]
        metric_df = metric_df.loc[best_idxs]
        ranks = metric_df.rank(axis = 0, ascending=False, method="average")
        metric_by_ds["rank"] = ranks.mean(axis=1)
        metric_by_ds["rank_std"] = ranks.std(axis=1)
    
    metric_by_ds = metric_by_ds.sort_values(by=['rank', 'rank_std'], ascending=[True, True])

    metric_by_ds = metric_by_ds[[*[de for d in tasks for de in [d, d + "_std"]], *rank_columns]]
    # metric_by_ds.to_csv(f"{base_path}/{metric_name}{suffix}-avg.csv")

    values_to_highlight = metric_by_ds[tasks].to_numpy().max(axis=0)

    rows = []
    for row_id, row in enumerate(metric_by_ds.reset_index().to_dict(orient="records")):
        new_row = {}
        infl_method = row["infl"]
        agg_method = row["agg"]
        layer = row["layer"]
        module = row["module"].replace("embed_tokens", "WE")
        if with_row_id:
            new_row["id"] = (row_id + 1)
        new_row["infl"] = infl_method
        new_row["agg"] = agg_method
        find_layer = re.match(r"layers\s(\d+)\s", module)
        if find_layer is not None:
            layer = find_layer.group(1)
            module = module.replace(find_layer.group(0), "")
        new_row["layer"] = "WE" if module == "WE" else layer
        new_row["module"] = module
        for did, d in enumerate([*tasks, "rank"]):
            should_highlight = False
            if d == "rank":
                m = round(row[d] * 10) / 10
                m_std = round(row[d + "_std"] * 10) / 10
            else:
                should_highlight = row[d] == values_to_highlight[did]
                m = round(row[d] * 1000) / 10
                m_std = round(row[d + "_std"] * 1000) / 10
            m = str(m).rstrip("0").rstrip(".").lstrip("0").replace("-0.", "-.")
            m_std = str(m_std).rstrip("0").rstrip(".").lstrip("0")

            if should_highlight:
                if m_std == "":
                    new_row[d] = f"\\textbf{{{m}}}"
                else:
                    new_row[d] = f"\\textbf{{{m}}} $\pm$ {m_std}"
            else:
                if m_std == "":
                    new_row[d] = m
                else:
                    new_row[d] = f"{m} $\pm$ {m_std}"
        rows.append(new_row)

    with open(f"{base_path}/ndr-{metric_name}{suffix}{custom_suffix}-avg.tex", "w") as stats_file:
        s = tabulate(rows, headers = "keys", showindex=False, tablefmt="latex")
        s = s.replace("\\textbackslash{}", "\\").replace("\\$", "$").replace("hf\_we\_topk\_10", "hf$^{10}_{we}$").replace("hf\_we\_", "hf$_{we}$").replace("\\_", "_").replace("\{", "{").replace("\}", "}").replace("\^{}", "^").replace("lllllllllll", "ll|ccccccccc").replace("rand", "\\hline rand")\
            .replace("_10", "$^{10}$").replace("_50", "$^{50}$")\
            .replace("commonset", "cset").replace("-20", "$^{20}$").replace("-30", "$^{30}$")\
            .replace("commonsubset", "csset").replace("out_proj", "proj")\
            .replace('self_attn ', '')\
            .replace('v_proj', 'value').replace('q_proj', 'query')
        print(s, file = stats_file)

    pass

In [12]:
dss = ["mrpc", "qnli", "sst2", "qqp", "cola", "mnli", "rte", "stsb"]

process_ndr_table("./", tasks=dss, with_row_id=False, custom_suffix = "-agga", 
                    best_group_by=["infl", "agg"], 
                #   layers=selected_layers, 
                    ndr_prefix="ndr_agga",
                #   agg_method_names=agg_method_names
                    )

/scratch/local/18603686/ipykernel_3528672/1480102731.py:37: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  metric_by_ds_mean = metric_df.groupby(level=1, axis=1).mean()
/scratch/local/18603686/ipykernel_3528672/1480102731.py:38: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  metric_by_ds_std = metric_df.groupby(level=1, axis=1).std()
/scratch/local/18603686/ipykernel_3528672/1480102731.py:46: FutureWarning: The 'axis' keyword in DataFrame.groupby is deprecated and will be removed in a future version.
  best_idxs = metric_by_ds.groupby(level=best_group_by, axis=0)['rank'].idxmin()


In [2]:
import numpy as np
import torch
from datasets import load_from_disk
import pandas as pd

/blue/anshumanc.usf/nn-infl/nn-infl-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
%cd /blue/anshumanc.usf/nn-infl/qwen

/blue/anshumanc.usf/nn-infl/qwen


In [3]:
!ls

1-init-0.out	  4-score-vk-4.out
1-init-1.out	  4-score-vk-5.out
1-init-2.out	  4-score-vk-6.out
1-init-3.out	  4-score-vk-7.out
1-init-4.out	  4-tlogit-0.out
1-init-5.out	  4-tlogit-1.out
1-init-6.out	  4-tlogit-2.out
1-init-7.out	  4-tlogit-3.out
2-tun-0.out	  4-tlogit-4.out
2-tun-1.out	  4-tlogit-5.out
2-tun-2.out	  4-tlogit-6.out
2-tun-3.out	  4-tlogit-7.out
2-tun-4.out	  5-tun2-0.out
2-tun-5.out	  5-tun2-1.out
2-tun-6.out	  5-tun2-2.out
2-tun-7.out	  5-tun2-3.out
3-infl-0.out	  5-tun2-4.out
3-infl-1.out	  5-tun2-5.out
3-infl-2.out	  5-tun2-6.out
3-infl-3.out	  5-tun2-7.out
3-infl-4.out	  cancellation_m_bl.pkl
3-infl-5.out	  cola
3-infl-6.out	  core.python-11-1763564569-10081-3337-c1004a-s15.ufhpc.4186821
3-infl-7.out	  finetuning.out
4-cagg-0.out	  groups.json
4-cancel-0.out	  infl_layer_ranks.csv
4-cancel-1.out	  mnli
4-cancel-2.out	  mrpc
4-cancel-3.out	  ndr-30-agga-avg.tex
4-cancel-4.out	  ndr_agga_cola.pcl
4-cancel-5.out	  ndr_agga_mnli.pcl
4-cancel-6.out	  ndr_agga_mrpc.pcl
4

In [7]:
def extract_qv(name: str) -> str:
    if ".q_proj" in name or ".query." in name:
        return "q"
    elif ".v_proj" in name or ".value." in name:
        return "v"
    return ""

def extract_module(name: str) -> str: # either lora A or B
    if "lora_A" in name:
        return f"A {extract_qv(name)}"
    elif "lora_B" in name:
        return f"B {extract_qv(name)}"
    elif (".word_embeddings." in name) or (name == 'word_embeddings') or (".embed_tokens." in name):
        return "WE"
    elif (".classifier." in name) or (name == 'classifier') or (".score." in name):
        return "CL"
    raise ValueError(f"Unknown module name {name}")

def extract_layer(name:str) -> int: # numeric
    name_parts = name.split('.')
    loc_of_layer = name_parts.index("layers") if "layers" in name_parts else -1
    if loc_of_layer < 0:
        loc_of_layer = name_parts.index("layer") if "layer" in name_parts else -1
    if loc_of_layer > 0 and loc_of_layer + 1 < len(name_parts):
        layer = name_parts[loc_of_layer + 1]
        return int(layer)
    raise ValueError(f"Unknown layer {name}")

In [6]:
def rank_matrix_score(int_matrix: torch.Tensor, *, 
                        use_correct = None, correct_infl_preds = None, 
                        chunk_size = 10000, rank_score_fn = lambda x,**_:x, **_):
    ''' 3-D tensor: train sample * module * infl sample '''
    if use_correct is not None:
        if use_correct:
            int_matrix = int_matrix[:, :, correct_infl_preds]
        else:
            int_matrix = int_matrix[:, :, ~correct_infl_preds]    
    
    total_int_matrix = int_matrix.view(int_matrix.shape[0], -1)
    ranks = torch.zeros_like(total_int_matrix, dtype=torch.float, device = total_int_matrix.device)
    rank_ranges = torch.arange(total_int_matrix.shape[0], device = total_int_matrix.device, dtype=torch.float).view(-1, 1).repeat(1, chunk_size)
    for (int_matrix_view, ranks_view) in zip(torch.split(total_int_matrix, chunk_size, dim = 1),
                                                torch.split(ranks, chunk_size, dim = 1)):
        sort_indexes = torch.argsort(int_matrix_view, dim = 0)
        ranks_view.scatter_(0, sort_indexes, rank_ranges)
        del sort_indexes
    del rank_ranges

    scores = rank_score_fn(ranks.view(int_matrix.shape), correct_infl_preds = correct_infl_preds)
    del ranks
    return scores



In [38]:
model = "roberta"
tasks = "qnli,mrpc,sst2,qqp,cola,mnli,rte,stsb"
methods = "hf,cos,datainf"
device = "cuda"
num_layers_per_model = {"roberta":24,"qwen":28}
num_layers = num_layers_per_model[model]
cwd = f"/blue/anshumanc.usf/nn-infl/{model}"

In [23]:
import os
from scipy import stats

In [25]:
def mean_confidence(x, confidence_level = 0.95):
    mean = x.mean()
    n = len(x)
    df = n - 1
    se = stats.sem(x)  # same as std/sqrt(n)

    # t-based confidence interval
    ci_low, ci_high = stats.t.interval(confidence_level, df, loc=mean, scale=se)    
    return {
        "mean": mean,
        "ci_low": ci_low,
        "ci_high": ci_high
    }   

In [40]:
def infl_ranks(tasks = 'qnli,mrpc', 
               methods = "hf,hf_we_,hw_we_topk_10,cos,cov,datainf_one,datainf", 
               seeds="0,1,2,3,4,5,6,7,8,9",
               i_prefix='i_bl',
               device='cuda',
               num_layers: int = 16
               ):
    ''' In groups of noise/benign computes most influential training samples according to avg ranks over layers,
        and then records how ranks of these samples vary from layer to layer. 
        Also, does same with ``avg'' noise/benign samples 
    '''

    tasks = tasks.split(',')
    methods = methods.split(',')
    seeds = [int(s) for s in seeds.split(',')]

    layer_columns = list(range(num_layers + 2))
    df = pd.DataFrame(columns=['task', 'method', 'seed', 'sample_type', *layer_columns])
    df = df.set_index(['task', 'method', 'seed', 'sample_type'])

    for task in tasks:
        for seed in seeds:
            dataset_path = os.path.join(cwd, task, f'd_{task}_{seed}')

            datasets = load_from_disk(dataset_path) # add validation dataset 

            trainset = datasets['train']
            noise_mask = np.array(trainset['noise']) == 1
            noise_ids = np.where(noise_mask)[0].tolist()
            benign_ids = np.where(~noise_mask)[0].tolist()

            print(f"Train set size: {len(trainset)}, noise: {len(noise_ids)}, benign: {len(benign_ids)}, valset: {len(datasets['infl'])}")

            for method in methods:
                infl_score_path = os.path.join(cwd, task, f"{i_prefix}_{method}_{task}_{seed}.pt")
                infl_scores = torch.load(infl_score_path) #, map_location=torch.device(device))
                per_layer_rank_sums = {}
                for module_name, module_scores in infl_scores.items():
                    module_type = extract_module(module_name)
                    if module_type == "WE":
                        module_layer = 0
                    elif module_type == "CL":
                        module_layer = num_layers + 1
                    else:
                        module_layer = extract_layer(module_name) + 1
                    module_scores_t = module_scores.t() # IMPORTANT all aggs accepts train samples in rows
                    assert module_scores_t.ndim == 2
                    ranks = rank_matrix_score(module_scores_t, rank_score_fn = lambda x, **_: x)
                    if module_layer not in per_layer_rank_sums:
                        per_layer_rank_sums[module_layer] = [ranks]
                    else:
                        per_layer_rank_sums[module_layer].append(ranks)
                    pass
                # total_layer_counts = len(per_layer_rank_sums)
                # print("Total layer counts:", total_layer_counts)
                # total_rank_sums = None
                layer_medians = []
                # total_rank_count = 0
                layer_ranks = {}
                for k, v in per_layer_rank_sums.items():
                    # v_sum = torch.stack(v, dim=0).sum(dim=0)
                    v_sum = torch.cat(v, dim=1) # for each train sample all judjements in one row for a layer
                    # v_mean = v_sum / len(v)
                    v_median = torch.median(v_sum, dim=1).values
                    layer_ranks[k] = v_median #torch.mean(v_mean, dim=1)
                    layer_medians.append(v_median)
                    del v_sum
                    # total_rank_count += len(v)
                    # if total_rank_sums is None:
                    #     total_rank_sums = v_sum.clone()
                    # else:
                    #     total_rank_sums += v_sum
                    # del v_sum
                # total_rank_sums /= total_rank_count
                all_ranks = torch.stack(layer_medians, dim=1)
                # global_avg_ranks = torch.mean(total_rank_sums, dim=1)
                global_avg_ranks = torch.median(all_ranks, dim=1).values
                del all_ranks
                gloabl_noise_ranks = global_avg_ranks[noise_ids]
                gloabl_benign_ranks = global_avg_ranks[benign_ids]

                most_infl_noise_id_id = torch.argmax(gloabl_noise_ranks).item()
                most_infl_noise_id = noise_ids[most_infl_noise_id_id]
                most_infl_noise_rank = global_avg_ranks[most_infl_noise_id].item()
                print(f"[{task},{method},{seed}] Infl noise id {most_infl_noise_id}, rank {most_infl_noise_rank}")

                least_infl_noise_id_id = torch.argmin(gloabl_noise_ranks).item()
                least_infl_noise_id = noise_ids[least_infl_noise_id_id]
                least_infl_noise_rank = global_avg_ranks[least_infl_noise_id].item()
                print(f"[{task},{method},{seed}] Least infl noise id {least_infl_noise_id}, rank {least_infl_noise_rank}")

                most_infl_benign_id_id = torch.argmax(gloabl_benign_ranks).item()
                most_infl_benign_id = benign_ids[most_infl_benign_id_id]
                most_infl_benign_rank = global_avg_ranks[most_infl_benign_id].item()
                print(f"[{task},{method},{seed}] Infl benign id {most_infl_benign_id}, rank {most_infl_benign_rank}")

                least_infl_benign_id_id = torch.argmin(gloabl_benign_ranks).item()
                least_infl_benign_id = benign_ids[least_infl_benign_id_id]
                least_infl_benign_rank = global_avg_ranks[least_infl_benign_id].item()
                print(f"[{task},{method},{seed}] Least infl benign id {least_infl_benign_id}, rank {least_infl_benign_rank}")

                res = {"top_noise": [0] * (num_layers + 2),
                    "top_benign": [0] * (num_layers + 2),
                    "avg_noise_median": [0] * (num_layers + 2),
                    "avg_noise_mean": [0] * (num_layers + 2),
                    "avg_noise_min": [0] * (num_layers + 2),
                    "avg_noise_max": [0] * (num_layers + 2),
                    "avg_benign_median": [0] * (num_layers + 2),
                    "avg_benign_mean": [0] * (num_layers + 2),
                    "avg_benign_min": [0] * (num_layers + 2),
                    "avg_benign_max": [0] * (num_layers + 2),
                    "least_noise": [0] * (num_layers + 2),
                    "least_benign": [0] * (num_layers + 2)
                    }
                for layer_id, layer_r in layer_ranks.items():
                    most_infl_noise_rank = layer_r[most_infl_noise_id].item()
                    most_infl_benign_rank = layer_r[most_infl_benign_id].item()
                    least_infl_noise_rank = layer_r[least_infl_noise_id].item()
                    least_infl_benign_rank = layer_r[least_infl_benign_id].item()
                    avg_noise_median = torch.median(layer_r[noise_ids]).item()
                    avg_noise_rank_stats = mean_confidence(layer_r[noise_ids].cpu().numpy())
                    avg_bening_median = torch.median(layer_r[benign_ids]).item()
                    avg_benign_rank_stats = mean_confidence(layer_r[benign_ids].cpu().numpy())
                    res["top_noise"][layer_id] = most_infl_noise_rank
                    res["top_benign"][layer_id] = most_infl_benign_rank
                    res["avg_noise_median"][layer_id] = avg_noise_median
                    res["avg_noise_mean"][layer_id] = avg_noise_rank_stats["mean"]
                    res["avg_noise_min"][layer_id] = avg_noise_rank_stats["ci_low"]
                    res["avg_noise_max"][layer_id] = avg_noise_rank_stats["ci_high"]
                    res["avg_benign_median"][layer_id] = avg_bening_median
                    res["avg_benign_mean"][layer_id] = avg_benign_rank_stats["mean"]
                    res["avg_benign_min"][layer_id] = avg_benign_rank_stats["ci_low"]
                    res["avg_benign_max"][layer_id] = avg_benign_rank_stats["ci_high"]
                    res["least_noise"][layer_id] = least_infl_noise_rank
                    res["least_benign"][layer_id] = least_infl_benign_rank
                    
                for k, v in res.items():
                    df.loc[(task, method, seed, k)] = v
    ranks_path = os.path.join(cwd, f"infl_layer_ranks-median.csv")
    df.to_csv(ranks_path)

    pass    

In [41]:
infl_ranks(tasks=tasks, methods=methods, device=device, num_layers=num_layers)


Train set size: 4500, noise: 900, benign: 3600, valset: 500
[qnli,hf,0] Infl noise id 4115, rank 3857.0
[qnli,hf,0] Least infl noise id 4459, rank 378.0
[qnli,hf,0] Infl benign id 3126, rank 3741.0
[qnli,hf,0] Least infl benign id 2008, rank 405.0
[qnli,cos,0] Infl noise id 3456, rank 2690.0
[qnli,cos,0] Least infl noise id 949, rank 1372.0
[qnli,cos,0] Infl benign id 1151, rank 2792.0
[qnli,cos,0] Least infl benign id 503, rank 1498.0
[qnli,datainf,0] Infl noise id 4115, rank 3707.0
[qnli,datainf,0] Least infl noise id 4459, rank 428.0
[qnli,datainf,0] Infl benign id 1873, rank 3586.0
[qnli,datainf,0] Least infl benign id 2008, rank 448.0
Train set size: 4500, noise: 900, benign: 3600, valset: 500
[qnli,hf,1] Infl noise id 3124, rank 3734.0
[qnli,hf,1] Least infl noise id 3986, rank 590.0
[qnli,hf,1] Infl benign id 3832, rank 3830.0
[qnli,hf,1] Least infl benign id 2331, rank 597.0
[qnli,cos,1] Infl noise id 2527, rank 2560.0
[qnli,cos,1] Least infl noise id 1919, rank 1610.0
[qnli,co